# S8 · Lab 01 — El Muro de la Linealidad (y el fracaso de la intuición)

## Propósito
Demostrar que el problema rara vez es “el algoritmo” y casi siempre es **la representación**.

Vas a:
1) forzar un modelo lineal hasta que falle,  
2) intentar “arreglarlo” a mano con features polinómicas (grado 2 y 3),  
3) observar por qué ese camino **no escala**.

> Este laboratorio es **visual**: dibujaremos nubes de puntos y fronteras de decisión.

---

## Qué vas a entregar (al final)
- Capturas/observaciones de 3 fronteras:
  - Baseline lineal (x1, x2)
  - Curvado manual (grado 2)
  - Curvado manual (grado 3)
- Respuestas breves a las preguntas de reflexión.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Paso 1 — El escenario del “crimen”
Creamos un mundo donde las reglas “rectas” fallan: dos lunas entrelazadas con ruido.


In [ ]:
# Generamos las lunas
X, y = make_moons(n_samples=1200, noise=0.25, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape

In [ ]:
# Visualización: ¿podrías separar esto con una sola regla recta?
plt.figure()
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train)
plt.title("¿Separarías esto con una sola línea recta?")
plt.xlabel("Característica A (x1)")
plt.ylabel("Característica B (x2)")
plt.show()

**Pregunta para ti (rápida, sin código):**  
Si tu frontera tuviera que ser una línea recta… ¿cuántos puntos crees que dejarías mal?
(Escribe una estimación mental. Luego lo comprobamos.)


## Función auxiliar — Dibujar la frontera de decisión
Esta función:
- crea una rejilla en el plano,
- pide al modelo que prediga,
- pinta el “mapa” de clases y encima los puntos reales.

> No la modifiques salvo que lo necesites.


In [ ]:
def plot_decision_boundary(predict_fn, X, y, title, pad=0.6, step=0.02):
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step),
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = predict_fn(grid).reshape(xx.shape)

    plt.figure()
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", linewidths=0.2)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

In [ ]:
def plot_misclassified(X, y_true, y_pred, title):
    wrong = (y_true != y_pred)
    plt.figure()
    plt.scatter(X[:,0], X[:,1], c=y_true, edgecolors="k", linewidths=0.2, alpha=0.65)
    # marcamos errores con un círculo grande
    plt.scatter(X[wrong,0], X[wrong,1], facecolors="none", edgecolors="red", linewidths=1.8, s=120, label="Mal clasificados")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    if wrong.any():
        plt.legend()
    plt.show()

## Paso 2 — Baseline: la apuesta lineal
Entrenamos una Regresión Logística con las variables originales (`x1`, `x2`).

Por definición, su frontera es lineal en el espacio original.


In [ ]:
clf_lin = LogisticRegression(max_iter=2000)
clf_lin.fit(X_train, y_train)

pred_lin = clf_lin.predict(X_test)
acc_lin = accuracy_score(y_test, pred_lin)

print(f"Resultado baseline lineal: {acc_lin:.3f}")

In [ ]:
plot_misclassified(
    X_test, y_test, pred_lin,
    title="Baseline (test) — errores resaltados (círculos rojos)"
)

In [ ]:
plot_decision_boundary(
    predict_fn=lambda grid: clf_lin.predict(grid),
    X=X_train,
    y=y_train,
    title="Baseline (LogisticRegression) — frontera lineal en (x1,x2)"
)

### Observación guiada
- ¿Qué parte de la nube “corta por la mitad” la frontera?
- ¿En qué zonas parece inevitable equivocarse si la frontera es recta?


## Paso 3 — El “truco” del ingeniero (grado 2)
Como humanos, intentamos “curvar” el espacio dándole al modelo nuevas variables:
- `x1²`, `x2²`, `x1·x2`

La idea es: “si la frontera necesita curvarse, le damos curvas”.


In [ ]:
# Features manuales grado 2 (mínimo)
x1_tr, x2_tr = X_train[:, 0], X_train[:, 1]
x1_te, x2_te = X_test[:, 0], X_test[:, 1]

Xtr_g2 = np.column_stack([x1_tr, x2_tr, x1_tr**2, x2_tr**2, x1_tr * x2_tr])
Xte_g2 = np.column_stack([x1_te, x2_te, x1_te**2, x2_te**2, x1_te * x2_te])

clf_g2 = LogisticRegression(max_iter=2000)
clf_g2.fit(Xtr_g2, y_train)

pred_g2 = clf_g2.predict(Xte_g2)
acc_g2 = accuracy_score(y_test, pred_g2)

print(f"Accuracy con variables de Grado 2: {acc_g2:.3f}")

In [ ]:
plot_misclassified(
    X_test, y_test, pred_g2,
    title="Grado 2 (test) — errores resaltados (círculos rojos)"
)

In [ ]:
def predict_g2(grid):
    g1, g2 = grid[:, 0], grid[:, 1]
    grid_aug = np.column_stack([g1, g2, g1**2, g2**2, g1*g2])
    return clf_g2.predict(grid_aug)

plot_decision_boundary(
    predict_fn=predict_g2,
    X=X_train,
    y=y_train,
    title="Curvado manual (Grado 2) — misma LogisticRegression, otra representación"
)

### Momento de la verdad (importante)
Si el grado 2 apenas mejora (o incluso empeora), NO es un error del alumno.

Es el aprendizaje del laboratorio:

> Tu intuición de “meter un cuadrado” puede ser insuficiente  
> para una forma entrelazada (tipo “S”).

**Pregunta de parada obligatoria:**  
¿Por qué una transformación de grado 2 puede seguir siendo demasiado rígida aquí?


## Paso 4 — Forzando la máquina (grado 3)
Si el grado 2 no basta, subimos a grado 3.

Esto ya implica:
- más columnas,
- más combinaciones,
- más esfuerzo humano para “inventar el espacio correcto”.


In [ ]:
# Features manuales grado 3 (selección representativa)
Xtr_g3 = np.column_stack([
    x1_tr, x2_tr,
    x1_tr**2, x2_tr**2, x1_tr*x2_tr,
    x1_tr**3, x2_tr**3,
    (x1_tr**2)*x2_tr, x1_tr*(x2_tr**2)
])

Xte_g3 = np.column_stack([
    x1_te, x2_te,
    x1_te**2, x2_te**2, x1_te*x2_te,
    x1_te**3, x2_te**3,
    (x1_te**2)*x2_te, x1_te*(x2_te**2)
])

clf_g3 = LogisticRegression(max_iter=3000)
clf_g3.fit(Xtr_g3, y_train)

pred_g3 = clf_g3.predict(Xte_g3)
acc_g3 = accuracy_score(y_test, pred_g3)

print(f"Accuracy con variables de Grado 3: {acc_g3:.3f}")

In [ ]:
plot_misclassified(
    X_test, y_test, pred_g3,
    title="Grado 3 (test) — errores resaltados (círculos rojos)"
)

In [ ]:
def predict_g3(grid):
    g1, g2 = grid[:, 0], grid[:, 1]
    grid_aug = np.column_stack([
        g1, g2,
        g1**2, g2**2, g1*g2,
        g1**3, g2**3,
        (g1**2)*g2, g1*(g2**2)
    ])
    return clf_g3.predict(grid_aug)

plot_decision_boundary(
    predict_fn=predict_g3,
    X=X_train,
    y=y_train,
    title="Curvado manual (Grado 3) — más expresividad, más coste humano"
)

## Comparación rápida (visual): accuracy vs “esfuerzo humano”
Aquí no buscamos “ganar”. Buscamos ver el patrón: **cada punto extra cuesta más decisiones humanas**.


In [ ]:
plt.figure()
plt.bar(["Lineal", "Grado 2", "Grado 3"], [acc_lin, acc_g2, acc_g3])
plt.title("Accuracy en test según la representación")
plt.xlabel("Representación (misma LogisticRegression)")
plt.ylabel("Accuracy")
plt.show()

## Conclusión y reflexión final (conecta los puntos)

### Hechos observados
- Baseline: el modelo lineal falla porque su frontera es recta.
- Grado 2: “curvar” un poco puede no bastar (tu intuición puede fallar).
- Grado 3: puedes mejorar… pero pagas con más columnas y más complejidad manual.

### El límite del ingeniero (la idea clave)
Para ganar unos pocos puntos, has tenido que:
- inventar variables,
- escribir combinaciones,
- y repetir el ciclo de prueba/error.

### La pregunta del millón
Si en vez de 2 variables tuvieras 50 (píxeles, comportamiento de clientes, sensores):
- ¿podrías escribir a mano todas las combinaciones relevantes de grado 3?
- ¿y mantener ese sistema cuando los datos cambian?

### Puente a Redes Neuronales
Necesitamos un sistema que:
- fabrique “curvaturas” y combinaciones automáticamente,
- y las ajuste con el error como guía.

**Siguiente paso:** abre el Lab 02 para ver qué ocurre cuando intentamos automatizar esto “a lo bruto”.


## Preguntas de cierre (responde aquí)

1) ¿Qué ha cambiado realmente: el algoritmo o la representación?  
2) ¿Qué muestra el fracaso del grado 2 sobre “intuición geométrica” vs datos reales?  
3) ¿Cuál es el coste cognitivo de pasar de grado 2 a grado 3?  
4) Si tuvieras 100 variables, ¿qué parte del proceso se vuelve inviable y por qué?  
5) Con una frase: ¿por qué esto abre la puerta a redes neuronales?

